# 0503 · 00 · Preparar el entorno

Laboratorio 0503 · Columnas de metadatos durante la ingesta.

Ejecuta este notebook con **Run all** antes de empezar. Hace cuatro cosas:

1. Comprueba que los cuatro Parquet de usuarios están en la carpeta `datasets/users-historical/` del repositorio.
2. Crea el catálogo `lab0503` con dos schemas:
   - `bronze`: donde se crean las tablas del laboratorio.
   - `upload`: con el volumen `upload`, la carpeta de aterrizaje de los ficheros.
3. Deja el laboratorio en su **estado inicial**: borra las tablas de una ejecución anterior, vacía el volumen y copia los cuatro Parquet a `upload/users-historical/`.
4. Verifica, fichero a fichero, que el volumen contiene **251.501 filas** en total.

Si quieres repetir el laboratorio desde cero, vuelve a ejecutarlo: siempre deja el mismo punto de partida.

In [ ]:
%python
import os

CATALOGO = "lab0503"
VOLUMEN_UPLOAD = f"/Volumes/{CATALOGO}/upload/upload"
RUTA_ORIGEN = f"{VOLUMEN_UPLOAD}/users-historical"

# La carpeta de este notebook es la del laboratorio dentro de tu Git folder;
# los datasets viven un nivel más arriba, en la raíz del repositorio.
RUTA_LAB = os.getcwd()
if not RUTA_LAB.startswith("/Workspace"):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    RUTA_LAB = "/Workspace" + os.path.dirname(ctx.notebookPath().get())
RUTA_DATASETS = f"{os.path.dirname(RUTA_LAB)}/datasets/users-historical"

print("Laboratorio:", RUTA_LAB)
print("Datasets:   ", RUTA_DATASETS)
print("Catálogo:   ", CATALOGO)

## 1 · Comprobar los ficheros del repositorio

In [ ]:
%python
FILAS_ESPERADAS = {
    "part-00000.snappy.parquet": 62876,
    "part-00001.snappy.parquet": 62875,
    "part-00002.snappy.parquet": 62875,
    "part-00003.snappy.parquet": 62875,
}

if not os.path.isdir(RUTA_DATASETS):
    raise FileNotFoundError(
        f"No encuentro {RUTA_DATASETS}. Comprueba que tu Git folder está actualizado "
        "y que la carpeta datasets/ está en la raíz del repositorio.")

faltan = [f for f in FILAS_ESPERADAS if not os.path.exists(f"{RUTA_DATASETS}/{f}")]
if faltan:
    raise FileNotFoundError("Faltan ficheros en el repositorio: " + ", ".join(faltan))
print("Los cuatro Parquet están en el repositorio.")

## 2 · Catálogo, schemas y volumen

In [ ]:
%python
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for schema in ["bronze", "upload"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.upload.upload")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))

## 3 · Volver al estado inicial

Borra las tablas del laboratorio si existen de una ejecución anterior y vacía el volumen. Los ficheros se copian de nuevo en el paso siguiente, así que su **hora de modificación** (`file_modification_time`) será la de esta ejecución.

In [ ]:
%python
for tabla in ["historical_users_bronze", "historical_users_bronze_python_metadata"]:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOGO}.bronze.{tabla}")

for item in dbutils.fs.ls(VOLUMEN_UPLOAD):
    dbutils.fs.rm(item.path, recurse=True)
print("vaciado", VOLUMEN_UPLOAD)

## 4 · Cargar los Parquet y verificar

In [ ]:
%python
import shutil

os.makedirs(RUTA_ORIGEN, exist_ok=True)
for fichero in FILAS_ESPERADAS:
    shutil.copyfile(f"{RUTA_DATASETS}/{fichero}", f"{RUTA_ORIGEN}/{fichero}")
    print("copiado", fichero)

recuento = {
    fila["fichero"]: fila["filas"]
    for fila in spark.sql(f"""
        SELECT _metadata.file_name AS fichero, count(*) AS filas
        FROM read_files('{RUTA_ORIGEN}', format => 'parquet')
        GROUP BY ALL
    """).collect()
}

errores = [f"{f}: {recuento.get(f)} filas, esperaba {n}"
           for f, n in FILAS_ESPERADAS.items() if recuento.get(f) != n]
if errores:
    raise ValueError("El volumen no tiene los datos esperados: " + "; ".join(errores))

total = sum(recuento.values())
print(f"Entorno listo: {RUTA_ORIGEN} contiene 4 Parquet con {total} filas.")